# 01 CDRMIP Pr Pipeline

In [1]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from cdrmip_extremes import (
        configs, pipeline, io_disk,
        file_search, processing,
        load_raw, extremes, utils, stats, plotting
)
from cdrmip_extremes.extremes_pipeline import ExtremesPipeline
from cdrmip_extremes.configs import PROC_DIR

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
config = pipeline.PipelineConfig(
    experiments=[
        pipeline.ExperimentConfig(
            experiment_id='1pctCO2-cdr', table_id='day', variable_id='pr',
            anomaly_reference='picontrol',grid_label=None,
            crossing_gwls=[1.5], overshoot=True,
            gwl_match_period=(320, 340),
            rolling_window=21,
        ),
    ],
    realign_years_overrides={'1pctCO2': (0, 140)}
)

In [4]:
config = pipeline.CDRMIPPipeline(config)

In [5]:
pipe = config.load_raw()

=== Loading 1pctCO2-cdr ===
Searching for files matching: {'experiment_id': '1pctCO2-cdr', 'table_id': 'day', 'variable_id': 'pr', 'grid_label': None, 'member_id': None}
Loading ACCESS-ESM1-5 / r1i1p1f1
  0241-01-01 12:00:00 -> 1000-12-31 12:00:00
Loading CESM2 / r1i1p1f1
  0001-01-01 00:00:00 -> 0200-12-31 00:00:00
Loading CNRM-ESM2-1 / r1i1p1f2
  1990-01-01 12:00:00 -> 2189-12-31 12:00:00
Loading CanESM5 / r1i1p2f1
  1991-01-01 12:00:00 -> 2290-12-31 12:00:00
Loading GFDL-ESM4 / r1i1p1f1
  0141-01-01 12:00:00 -> 0340-12-31 12:00:00
Loading MIROC-ES2L / r1i1p1f2
  1989-01-01 12:00:00 -> 2490-12-31 12:00:00
Loading NorESM2-LM / r1i1p1f1
  0141-01-01 12:00:00 -> 0399-12-31 12:00:00
Loading UKESM1-0-LL / r1i1p1f2
  1990-01-01 12:00:00 -> 2639-12-30 12:00:00


In [ ]:
pipe.concatenate_to_parents()

Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 1pctCO2 r1i1p1f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [ACCESS-ESM1-5] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for CESM2 1pctCO2-cdr r1i1p1f1 parent
Loading CESM2 1pctCO2 r1i1p1f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [CESM2] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 00:00:00, child relabeled to start 0140-01-01 00:00:00)
Searching for CNRM-ESM2-1 1pctCO2-cdr r1i1p1f2 parent
Loading CNRM-ESM2-1 1pctCO2 r1i1p1f2
Concatenating 1pctCO2-cdr -> 1pctCO2 [CNRM-ESM2-1] via realign_years=(0, 140)


In [ ]:
pipe.load_picontrol()

In [ ]:
pipe.regrid()

## Calculate metrics

In [ ]:
pipe.compute_rxnday()

In [ ]:
pipe.compute_rxnday(n=5)

In [ ]:
pipe.compute_rxnday_anomaly()

In [ ]:
pipe.rxnday_anomaly[1]

# Save

In [ ]:
io_disk.save_tree(
    pipe.rxnday_anomaly[1],
    PROC_DIR,
    'rx1day',
    activity_id='CDRMIP',
    category='anomaly',
    overwrite=True
)

In [ ]:
io_disk.save_tree(
    pipe.rxnday_anomaly[5],
    PROC_DIR,
    'rx5day',
    activity_id='CDRMIP',
    category='anomaly',
    overwrite=True
)